[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/09.%20The%20Quay%20Crane%20Scheduling%20Problem/P9-Tier-9_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 9. The Quay Crane Scheduling Problem
## Tier 9 — QUBO Formulation and the Quantum Approximate Optimization Algorithm

This notebook formulates a simplified quay crane scheduling problem (QCSP) as a quadratic unconstrained binary optimization (QUBO) model and explains how the Quantum Approximate Optimization Algorithm (QAOA) uses that model. The example has four cranes and eight bays. Its QUBO has 32 binary variables.

A classical state-vector simulation of QAOA is included for small QUBOs. A state vector for this 32-variable instance would require \(2^{32}\) complex amplitudes, so the full simulation is not run for the example. The notebook reports that computational limit explicitly and uses exhaustive enumeration of crane assignments as a reference benchmark; the benchmark is not presented as a QAOA result.

### Learning goals
- Interpret binary assignment variables and construct a QUBO objective with penalty terms.
- Understand the QAOA state, cost unitary, mixer unitary, and parameter-grid search.
- Distinguish the QUBO objective from the makespan used to evaluate a decoded schedule.
- Understand the memory and computational limits of classical state-vector simulation.

### Inputs and outputs
The inputs are the crane and bay counts, bay positions and processing times, travel-cost coefficient, and penalty weights. The notebook constructs a QUBO matrix, defines a classical QAOA simulator for instances small enough to fit in memory, decodes binary assignments, and benchmarks schedule makespan against an exact reference and an LPT heuristic.

### Scope and modeling assumptions
Each bay is assigned to exactly one crane. Cranes are indexed in their physical left-to-right order, and a crossing penalty discourages assignments that reverse that order. For a crane assigned several bays, the schedule visits them in increasing bay position and starts at its first assigned bay; therefore, no travel before the first assigned bay is charged. The QUBO uses a pairwise-distance travel-cost proxy, whereas schedule makespan uses travel along this ordered route. These are distinct measures, so a low QUBO energy does not by itself imply a low makespan. Capacity, precedence, crane interference beyond the crossing penalty, and other operational constraints are outside this simplified model.

In [ ]:
# Dependencies are expected to be available in the notebook environment.
try:
    import itertools
    from dataclasses import dataclass
    from typing import Any, Dict, List, Optional, Tuple

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
except ImportError as error:
    raise ImportError(
        "Missing dependency. This notebook requires numpy, pandas, and matplotlib."
    ) from error

print("Dependencies imported successfully.")

## QUBO formulation for the QCSP

A binary decision variable records whether a particular crane receives a particular bay. The QUBO combines assignment costs with penalties that make constraint violations expensive. For a binary vector \(x\), the energy is represented as \(E(x)=x^{\mathsf T}Qx\), where \(Q\) is a symmetric matrix. Minimizing this energy is the classical optimization problem that QAOA is designed to approximate.

Let \(x_{ij}=1\) when crane \(i\) is assigned bay \(j\), and \(x_{ij}=0\) otherwise. Let \(t_j\) be the processing time of bay \(j\), \(p_j\) its position, \(v\) the travel-cost coefficient per unit of bay distance, and \(A,B>0\) the assignment and crossing penalty weights. The model used here is

$$
E(x)=\sum_{i,j} t_jx_{ij}
+\sum_i\sum_{j<l}v|p_j-p_l|x_{ij}x_{il}
+A\sum_j\left(\sum_i x_{ij}-1\right)^2
+B\sum_{i<k}\sum_{j>l}x_{ij}x_{kl}.
$$

The first term charges processing time. The second is a pairwise-distance proxy for travel when a crane receives multiple bays; it is not the ordered-route travel used later to calculate makespan. The third term is zero exactly when every bay is assigned to one crane. The last term penalizes a crossing: a lower-indexed crane assigned a higher-position bay while a higher-indexed crane is assigned a lower-position bay.

For binary variables, the assignment penalty expands as

$$
A\left(\sum_i x_{ij}-1\right)^2
= -A\sum_i x_{ij}+2A\sum_{i<k}x_{ij}x_{kj}+A.
$$

The constant \(A\) does not affect which assignment minimizes the energy, so it is omitted from the matrix. Because \(Q\) is symmetric and the energy is \(x^{\mathsf T}Qx\), a coefficient \(c\) for an unordered pair \(x_ax_b\) is stored as \(c/2\) in each of \(Q_{ab}\) and \(Q_{ba}\). This convention is used for both pairwise travel and penalty terms.

In [ ]:
# Instance definition: four cranes and eight bays.
@dataclass(frozen=True)
class Bay:
    id: int
    position: int
    process_time: int  # minutes


bays: List[Bay] = [
    Bay(id=0, position=0, process_time=45),
    Bay(id=1, position=1, process_time=30),
    Bay(id=2, position=2, process_time=25),
    Bay(id=3, position=3, process_time=35),
    Bay(id=4, position=4, process_time=40),
    Bay(id=5, position=5, process_time=20),
    Bay(id=6, position=6, process_time=55),
    Bay(id=7, position=7, process_time=28),
]

NUM_CRANES = 4
NUM_BAYS = len(bays)
TRAVEL_PER_BAY = 5.0  # minutes per unit of bay distance

# Flatten (crane, bay) into one binary-vector index.
var_index = {
    (crane, bay_index): crane * NUM_BAYS + bay_index
    for crane in range(NUM_CRANES)
    for bay_index in range(NUM_BAYS)
}
N_VARS = NUM_CRANES * NUM_BAYS

print(f"Number of binary variables: {N_VARS}")
print("Variable mapping (first few):", dict(list(var_index.items())[:6]))

## Constructing the QUBO matrix

The matrix \(Q\) stores the linear and quadratic coefficients of the energy function. Its diagonal entries hold linear terms. Off-diagonal entries hold half of each unordered-pair coefficient, because \(x^{\mathsf T}Qx\) counts both \(Q_{ab}x_ax_b\) and \(Q_{ba}x_bx_a\). The assignment penalty contributes a negative linear term and a positive pairwise term; the omitted constant does not change the minimizing assignment.

The penalty weights \(A=10\) and \(B=5\) are the values used in this example. A finite penalty weight does not automatically guarantee that every energy-minimizing bitstring satisfies every constraint, so feasibility is checked after decoding.

In [ ]:
def build_qubo(A: float = 10.0, B: float = 5.0) -> np.ndarray:
    """Build the symmetric QUBO matrix for the specified instance."""
    Q = np.zeros((N_VARS, N_VARS), dtype=float)

    # Processing time is a linear cost for every crane-bay assignment.
    for crane in range(NUM_CRANES):
        for bay_index in range(NUM_BAYS):
            index = var_index[(crane, bay_index)]
            Q[index, index] += bays[bay_index].process_time

    # Store each same-crane pairwise travel coefficient symmetrically.
    for crane in range(NUM_CRANES):
        for first_bay in range(NUM_BAYS):
            for second_bay in range(first_bay + 1, NUM_BAYS):
                first_index = var_index[(crane, first_bay)]
                second_index = var_index[(crane, second_bay)]
                travel_cost = TRAVEL_PER_BAY * abs(
                    bays[first_bay].position - bays[second_bay].position
                )
                Q[first_index, second_index] += travel_cost / 2
                Q[second_index, first_index] += travel_cost / 2

    # Expand A * (sum_crane x_crane,bay - 1)^2 for each bay.
    for bay_index in range(NUM_BAYS):
        indices = [
            var_index[(crane, bay_index)]
            for crane in range(NUM_CRANES)
        ]
        for index in indices:
            Q[index, index] -= A
        for first_position in range(NUM_CRANES):
            for second_position in range(first_position + 1, NUM_CRANES):
                first_index = indices[first_position]
                second_index = indices[second_position]
                Q[first_index, second_index] += A
                Q[second_index, first_index] += A

    # Penalize assignments that reverse the left-to-right crane order.
    for first_crane in range(NUM_CRANES):
        for second_crane in range(first_crane + 1, NUM_CRANES):
            for first_bay in range(NUM_BAYS):
                for second_bay in range(first_bay):
                    first_index = var_index[(first_crane, first_bay)]
                    second_index = var_index[(second_crane, second_bay)]
                    Q[first_index, second_index] += B / 2
                    Q[second_index, first_index] += B / 2

    return Q


Q = build_qubo(A=10.0, B=5.0)
assert Q.shape == (N_VARS, N_VARS)
assert np.allclose(Q, Q.T)
print("QUBO matrix shape:", Q.shape)
print("Sample diagonal coefficients:", np.diag(Q)[:8])

## Step 2 — Classical simulation of QAOA

QAOA alternates two unitary operations on a quantum state. The cost unitary adds a phase determined by the QUBO energy, and the mixer unitary rotates each binary variable so that probability amplitude can move among bitstrings. Starting from the equal superposition |+}^\(\otimes n\), a depth-{p} circuit applies \(p\) cost-mixer layer pairs. Classical parameter search selects values of {\(\gamma_\ell\)} and {\(\beta_\ell\)} that minimize the expected QUBO energy.

For a bitstring \(z\in\{0,1\}^n\), the cost unitary is \(U_C(\gamma)=\exp(-i\gamma C)\), where \(C\) is diagonal and \(C_{zz}=z^{\mathsf T}Qz\). The mixer is \(U_B(\beta)=\exp(-i\beta\sum_{r=1}^n X_r)\), where \(X_r\) flips bit \(r\). The layer sequence is

$$
|\psi(\boldsymbol{\gamma},\boldsymbol{\beta})\rangle
=\prod_{\ell=1}^{p}U_B(\beta_\ell)U_C(\gamma_\ell)|+\rangle^{\otimes n}.
$$

A state-vector simulator stores \(2^n\) complex amplitudes and requires work proportional to \(p n 2^n\) per parameter setting. This implementation therefore enforces a qubit limit rather than attempting an allocation that is infeasible for the 32-variable example. When it is run on a small enough instance, it performs an actual state-vector QAOA simulation; random bitstring search is not used as a substitute.

In [ ]:
def qaoa_classical(
    Q: np.ndarray,
    p: int = 2,
    gamma_grid: Optional[np.ndarray] = None,
    beta_grid: Optional[np.ndarray] = None,
    max_statevector_qubits: int = 16,
) -> Dict[str, Any]:
    """Run a grid-search QAOA state-vector simulation for a small QUBO."""
    if Q.ndim != 2 or Q.shape[0] != Q.shape[1]:
        raise ValueError("Q must be a square matrix.")
    if not np.allclose(Q, Q.T):
        raise ValueError("Q must be symmetric.")
    if p < 1:
        raise ValueError("p must be at least 1.")

    num_qubits = Q.shape[0]
    if num_qubits > max_statevector_qubits:
        raise ValueError(
            f"State-vector simulation requires 2^{num_qubits} amplitudes, "
            f"which exceeds the configured limit of {max_statevector_qubits} qubits."
        )

    if gamma_grid is None:
        gamma_grid = np.linspace(0.1, 2.0, 3)
    if beta_grid is None:
        beta_grid = np.linspace(0.1, 2.0, 3)
    if len(gamma_grid) == 0 or len(beta_grid) == 0:
        raise ValueError("Parameter grids must not be empty.")

    num_states = 1 << num_qubits
    basis_indices = np.arange(num_states, dtype=np.uint64)
    bit_positions = np.arange(num_qubits, dtype=np.uint64)
    basis_bits = (
        (basis_indices[:, None] >> bit_positions[None, :]) & 1
    ).astype(float)
    energies = np.einsum("bi,ij,bj->b", basis_bits, Q, basis_bits)

    def apply_mixer(state: np.ndarray, beta: float) -> np.ndarray:
        """Apply each single-qubit X rotation to the state amplitudes."""
        cosine = np.cos(beta)
        sine = np.sin(beta)
        for qubit in range(num_qubits):
            stride = 1 << qubit
            blocks = state.reshape(-1, 2 * stride)
            zero_amplitudes = blocks[:, :stride].copy()
            one_amplitudes = blocks[:, stride:].copy()
            blocks[:, :stride] = (
                cosine * zero_amplitudes - 1j * sine * one_amplitudes
            )
            blocks[:, stride:] = (
                cosine * one_amplitudes - 1j * sine * zero_amplitudes
            )
        return state

    best_expected_energy = float("inf")
    best_params = None
    best_bitstring = None
    best_bitstring_energy = None
    initial_state = np.full(
        num_states, 1 / np.sqrt(num_states), dtype=complex
    )

    for gammas in itertools.product(gamma_grid, repeat=p):
        for betas in itertools.product(beta_grid, repeat=p):
            state = initial_state.copy()
            for gamma, beta in zip(gammas, betas):
                state *= np.exp(-1j * gamma * energies)
                apply_mixer(state, beta)

            probabilities = np.abs(state) ** 2
            expected_energy = float(np.dot(probabilities, energies))
            if expected_energy < best_expected_energy:
                most_probable_index = int(np.argmax(probabilities))
                best_expected_energy = expected_energy
                best_params = (gammas, betas)
                best_bitstring = basis_bits[most_probable_index].astype(int)
                best_bitstring_energy = float(energies[most_probable_index])

    return {
        "expected_energy": best_expected_energy,
        "best_energy": best_bitstring_energy,
        "best_params": best_params,
        "best_bitstring": best_bitstring,
    }


qaoa_result = None
try:
    qaoa_result = qaoa_classical(
        Q,
        p=2,
        gamma_grid=np.linspace(0.5, 1.5, 3),
        beta_grid=np.linspace(0.5, 1.5, 3),
    )
except ValueError as error:
    print(f"QAOA state-vector run not performed: {error}")

if qaoa_result is not None:
    print(f"Best QAOA expected energy: {qaoa_result['expected_energy']:.2f}")
    print(f"Most-probable bitstring energy: {qaoa_result['best_energy']:.2f}")
    print(f"Best parameters (gamma): {qaoa_result['best_params'][0]}")
    print(f"Best parameters (beta): {qaoa_result['best_params'][1]}")
    print(f"Most-probable bitstring (first 16 bits): {qaoa_result['best_bitstring'][:16]}")

## Step 3 — Decode a bitstring into a schedule

The binary vector uses the index mapping {\(c\cdot\texttt{NUM\_BAYS}+b\)} for crane \(c\) and bay \(b\). Decoding collects the selected bays for each crane. Feasibility checks then verify that every bay is assigned once and that the selected assignments contain no crossing penalty. The schedule evaluation orders each crane's bays by position, adds processing time and travel between consecutive bays, and takes the largest crane completion time as the makespan.

In [ ]:
def decode_solution(bitstring: np.ndarray) -> Dict[int, List[int]]:
    """Decode a binary assignment vector into crane-to-bay lists."""
    if len(bitstring) != N_VARS:
        raise ValueError(f"Expected a bitstring of length {N_VARS}.")
    if not np.all(np.isin(bitstring, [0, 1])):
        raise ValueError("The assignment vector must contain only 0 and 1.")

    assignments = {crane: [] for crane in range(NUM_CRANES)}
    for crane in range(NUM_CRANES):
        for bay_index in range(NUM_BAYS):
            index = var_index[(crane, bay_index)]
            if bitstring[index] == 1:
                assignments[crane].append(bay_index)
    return assignments


def is_feasible_assignment(assignments: Dict[int, List[int]]) -> bool:
    """Check unique bay assignment and the modeled non-crossing rule."""
    assigned_bays = [
        bay_index
        for crane_bays in assignments.values()
        for bay_index in crane_bays
    ]
    if sorted(assigned_bays) != list(range(NUM_BAYS)):
        return False

    for first_crane in range(NUM_CRANES):
        for second_crane in range(first_crane + 1, NUM_CRANES):
            for first_bay in assignments[first_crane]:
                for second_bay in assignments[second_crane]:
                    if first_bay > second_bay:
                        return False
    return True


def makespan_from_assignments(
    assignments: Dict[int, List[int]],
) -> float:
    """Evaluate ordered-route makespan in minutes."""
    def crane_completion(bay_indices: List[int]) -> float:
        ordered_bays = sorted(
            bay_indices,
            key=lambda bay_index: bays[bay_index].position,
        )
        if not ordered_bays:
            return 0.0

        completion = 0.0
        current_position = bays[ordered_bays[0]].position
        for bay_index in ordered_bays:
            target_position = bays[bay_index].position
            completion += TRAVEL_PER_BAY * abs(
                current_position - target_position
            )
            completion += bays[bay_index].process_time
            current_position = target_position
        return completion

    if set(assignments) != set(range(NUM_CRANES)):
        raise ValueError("Assignments must contain every crane key.")
    return max(crane_completion(assignments[crane]) for crane in range(NUM_CRANES))


if qaoa_result is not None:
    qaoa_assignments = decode_solution(qaoa_result["best_bitstring"])
    print("Crane assignments (crane -> bay indices):")
    for crane, assigned_bays in qaoa_assignments.items():
        print(f"Crane {crane}: {assigned_bays}")
    print("Assignment feasible:", is_feasible_assignment(qaoa_assignments))
    if is_feasible_assignment(qaoa_assignments):
        makespan_qaoa = makespan_from_assignments(qaoa_assignments)
        print(f"QAOA schedule makespan (minutes): {makespan_qaoa:.1f}")
    else:
        makespan_qaoa = None
        print("The most-probable bitstring does not satisfy the modeled constraints.")
else:
    qaoa_assignments = None
    makespan_qaoa = None

## Step 4 — Benchmark against classical references

The exact reference enumerates every assignment of eight bays to four cranes, so it examines \(4^8\) assignment patterns. Each pattern is evaluated using the same ordered-route makespan function. The LPT heuristic processes bays in descending processing-time order and assigns each bay to the crane with the smallest accumulated processing time; its final schedule is evaluated with the route-based makespan function.

The exact search has \(O(C^B B)\) evaluation work for \(C\) cranes and \(B\) bays, apart from the cost of computing each schedule's makespan. QAOA results are included in the comparison only if the state-vector simulation was actually run and its decoded assignment is feasible. A QUBO energy and a schedule makespan are different quantities, so the benchmark compares schedules using makespan.

In [ ]:
def exact_optimum() -> Dict[str, Any]:
    """Enumerate all crane assignments and return a minimum-makespan schedule."""
    best_makespan = float("inf")
    best_assignments = None

    for choice_tuple in itertools.product(range(NUM_CRANES), repeat=NUM_BAYS):
        assignments = {crane: [] for crane in range(NUM_CRANES)}
        for bay_index, crane in enumerate(choice_tuple):
            assignments[crane].append(bay_index)

        makespan = makespan_from_assignments(assignments)
        if makespan < best_makespan:
            best_makespan = makespan
            best_assignments = assignments

    return {"makespan": best_makespan, "assignments": best_assignments}


def enhanced_lpt() -> Dict[str, Any]:
    """Assign longest-processing-time bays to the least-loaded crane."""
    sorted_bays = sorted(
        range(NUM_BAYS),
        key=lambda bay_index: bays[bay_index].process_time,
        reverse=True,
    )
    assignments = {crane: [] for crane in range(NUM_CRANES)}
    processing_load = {crane: 0.0 for crane in range(NUM_CRANES)}

    for bay_index in sorted_bays:
        selected_crane = min(
            range(NUM_CRANES),
            key=lambda crane: processing_load[crane],
        )
        assignments[selected_crane].append(bay_index)
        processing_load[selected_crane] += bays[bay_index].process_time

    return {
        "makespan": makespan_from_assignments(assignments),
        "assignments": assignments,
    }


exact_result = exact_optimum()
heuristic_result = enhanced_lpt()
print(f"Exact reference makespan (minutes): {exact_result['makespan']:.1f}")
print(f"Tier 2 LPT makespan (minutes): {heuristic_result['makespan']:.1f}")

comparison_rows = [
    {
        "Method": "Exact reference",
        "Makespan (minutes)": exact_result["makespan"],
    },
    {
        "Method": "LPT heuristic",
        "Makespan (minutes)": heuristic_result["makespan"],
    },
]

if makespan_qaoa is not None:
    comparison_rows.append(
        {
            "Method": "QAOA state-vector simulation",
            "Makespan (minutes)": makespan_qaoa,
        }
    )

comparison = pd.DataFrame(comparison_rows)
comparison["Gap vs Optimal (%)"] = (
    (comparison["Makespan (minutes)"] - exact_result["makespan"])
    / exact_result["makespan"]
    * 100
).round(1)
comparison

## Step 5 — Optional: AWS Braket availability

This cell checks whether the AWS Braket SDK is importable. It does not submit a quantum task or build a complete QAOA circuit. Running a hardware experiment requires a configured AWS account, credentials, a supported device, and a circuit that implements the QUBO cost and mixer operations.

In [ ]:
try:
    from braket.aws import AwsDevice  # noqa: F401
    from braket.circuits import Circuit  # noqa: F401
    from braket.parametric import FreeParameter  # noqa: F401
except ImportError:
    print("AWS Braket SDK is not available; no quantum task will be submitted.")
else:
    print(
        "AWS Braket SDK is available. This notebook checks availability only; "
        "it does not submit a quantum task."
    )

## Step 6 — Interpretation and limitations

### What can be concluded from this example?
The QUBO matrix encodes processing costs, a pairwise travel proxy, a one-crane-per-bay assignment penalty, and a non-crossing penalty. The exact reference and LPT heuristic produce schedules that can be compared using the explicitly defined route-based makespan. This 32-variable instance is too large for the configured state-vector simulator, so it produces no QAOA schedule and no QAOA-versus-exact performance claim.

### What QAOA does and does not establish
QAOA prepares a probability distribution over bitstrings by alternating cost and mixer unitaries. A classical state-vector simulator calculates those amplitudes directly and has memory growth proportional to \(2^n\); it is therefore useful for small demonstrations but is not a scalable simulation strategy. Quantum superposition alone does not guarantee a speedup, and any quantum advantage depends on the problem structure, circuit depth, hardware quality, measurement cost, and classical parameter optimization.

### Modeling limitations
The QUBO travel term sums pairwise distances between bays assigned to the same crane. The schedule evaluator instead charges travel along bays ordered by position. Consequently, the QUBO energy is not the schedule makespan, even when all assignment constraints are satisfied. The penalty weights are finite, so a low-energy bitstring may still violate constraints; feasibility must be checked after decoding. Real quay operations may also require capacity, timing, precedence, safety-separation, and operational uncertainty constraints that are not represented here.

### Natural extensions
Further work could study additional operational constraints, larger or structurally different instances, and QAOA circuit execution on suitable hardware. Any such study should retain explicit feasibility checks and distinguish the encoded objective from the operational performance measure.

## Step 7 — Positioning this tier alongside multi-agent methods

QUBO and QAOA provide a way to express binary combinatorial objectives and investigate them with alternating quantum cost and mixer operations. Multi-agent methods instead emphasize distributed decision-making and coordination among agents. These are different modeling and solution perspectives; neither is guaranteed to outperform the other for every scheduling instance.

A practical comparison should account for the problem encoding, constraint handling, computational resources, solution quality, and runtime. Quantum hardware access, noise, circuit depth, and the overhead of mapping a problem into a QUBO are important considerations. The small state-vector QAOA implementation in this notebook is an educational simulator, not evidence of quantum speedup or hardware performance.